# ATLAS — CIFAR-10: NOVEL vs Literatura (SE-Net / CBAM)

**Objetivo:** responder se os métodos NOVEL ATLAS (**multi/tri_scale blend**) superam mecanismos **publicados** (SE-Net, CBAM) e se **generalizam** além do FashionMNIST.

**Como usar:**
1. Runtime → Change runtime type → **GPU (A100/H100)**
2. Runtime → Run all
3. Aguarde ~45–90 min (depende da GPU); ranking na célula 6

**Head-to-head (8 configs × 3 seeds = 24 treinos):**

| Config | Tipo | Referência |
|--------|------|------------|
| baseline | régua | — |
| **tri_scale_ls_wc** | **NOVEL ATLAS** | exp-053 recordista |
| **multi_scale_ls_wc** | **NOVEL ATLAS** | exp-047 campeão |
| blend_ls_wc | NOVEL+RECOMB | stack local_blend |
| se_block | **LITERATURA** | Hu et al. SE-Net (2018) |
| se_ls_wc | LITERATURA+stack | SE + LS + warmup/cosine |
| cbam | **LITERATURA** | Woo et al. CBAM (2018) |
| cbam_ls_wc | LITERATURA+stack | CBAM + LS + warmup/cosine |

**Comparação justa:** mesmo SmallCNN, width×2, batch 256, bf16, mesmos seeds, steps calibrados por orçamento total.

**Pergunta que este notebook responde:**
> "Nosso blend multi-escala supera SE-Net e CBAM em dataset diferente?"

In [ ]:
# @title 1. GPU + orçamento de tempo
import torch

assert torch.cuda.is_available(), "GPU indisponível! Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | VRAM: {vram:.0f} GB | PyTorch: {torch.__version__}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")

# Ajuste aqui se quiser mais/menos tempo total
TARGET_MINUTES = 60   # orçamento total do batch (24 treinos)
N_CONFIGS, N_SEEDS = 8, 3

In [ ]:
# @title 2. CIFAR-10 residente na GPU
from torchvision import datasets, transforms

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

train_ds = datasets.CIFAR10("./data", train=True, download=True)
val_ds = datasets.CIFAR10("./data", train=False, download=True)

def to_gpu(ds):
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - MEAN) / STD
    y = torch.tensor(ds.targets, dtype=torch.long)
    return x.to(DEVICE), y.to(DEVICE)

print("Carregando CIFAR-10 na GPU...")
X_TRAIN, Y_TRAIN = to_gpu(train_ds)
X_VAL, Y_VAL = to_gpu(val_ds)
print(f"train {tuple(X_TRAIN.shape)} | val {tuple(X_VAL.shape)}")

In [ ]:
# @title 3. Modelos: NOVEL ATLAS + SE-Net + CBAM
import math, random, time
import torch.nn as nn
import torch.nn.functional as F

# --- NOVEL ATLAS blends ---
class LocalBlend(nn.Module):
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.dw = nn.Conv2d(channels, channels, kernel, padding=kernel//2, groups=channels, bias=False)
        nn.init.dirac_(self.dw.weight)
    def forward(self, x):
        local = self.dw(x)
        gate = torch.sigmoid(x.mean(dim=(2,3), keepdim=True))
        return gate * local + (1.0 - gate) * x

class MultiScaleBlend(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.dw5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        for m in (self.dw3, self.dw5): nn.init.dirac_(m.weight)
    def forward(self, x):
        local = 0.5 * self.dw3(x) + 0.5 * self.dw5(x)
        gate = torch.sigmoid(x.mean(dim=(2,3), keepdim=True))
        return gate * local + (1.0 - gate) * x

class TriScaleBlend(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.dw5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.dw7 = nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False)
        for m in (self.dw3, self.dw5, self.dw7): nn.init.dirac_(m.weight)
    def forward(self, x):
        local = (self.dw3(x) + self.dw5(x) + self.dw7(x)) / 3.0
        gate = torch.sigmoid(x.mean(dim=(2,3), keepdim=True))
        return gate * local + (1.0 - gate) * x

def make_mixing(name, ch):
    return {"local_blend": LocalBlend(ch,3), "multi_scale_blend": MultiScaleBlend(ch),
            "tri_scale_blend": TriScaleBlend(ch)}.get(name, nn.Identity())

# --- LITERATURA: SE-Net (Hu et al. 2018) ---
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: gate de canal via GAP + MLP."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid), nn.ReLU(inplace=True),
            nn.Linear(mid, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * w

# --- LITERATURA: CBAM (Woo et al. 2018) ---
class CBAM(nn.Module):
    """Channel + Spatial attention."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.ca_fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sa_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        b, c, _, _ = x.shape
        avg = x.mean(dim=(2, 3))
        mx = x.view(b, c, -1).max(dim=2).values
        scale = torch.sigmoid(self.ca_fc(avg) + self.ca_fc(mx)).view(b, c, 1, 1)
        x = x * scale
        avg_map = x.mean(dim=1, keepdim=True)
        max_map = x.max(dim=1, keepdim=True)[0]
        scale = torch.sigmoid(self.sa_conv(torch.cat([avg_map, max_map], dim=1)))
        return x * scale

def make_attention(name, ch):
    if name == "se": return SEBlock(ch)
    if name == "cbam": return CBAM(ch)
    return nn.Identity()

class CIFARSmallCNN(nn.Module):
    """SmallCNN para CIFAR-10 (32x32x3 → 8x8 após 2 pools)."""
    def __init__(self, mixing="none", attention="none", width_mult=2.0, hidden_dim=256, dropout=0.1):
        super().__init__()
        c1, c2 = int(32*width_mult), int(64*width_mult)
        self.conv1 = nn.Conv2d(3, c1, 3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.mix1 = make_mixing(mixing, c1)
        self.mix2 = make_mixing(mixing, c2)
        self.att1 = make_attention(attention, c1)
        self.att2 = make_attention(attention, c2)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(c2 * 8 * 8, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 10)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
        self.mix1 = make_mixing(mixing, c1)
        self.mix2 = make_mixing(mixing, c2)

    def _block(self, conv, mix, att, x):
        h = self.act(conv(x))
        h = mix(h)
        h = att(h)
        return self.pool(h)

    def forward(self, x):
        x = self._block(self.conv1, self.mix1, self.att1, x)
        x = self._block(self.conv2, self.mix2, self.att2, x)
        x = self.dropout(self.act(self.fc1(x.flatten(1))))
        return self.fc2(x)

def lr_at_step(base_lr, step, total, warmup, scheduler):
    if warmup > 0 and step < warmup:
        return base_lr * (step + 1) / warmup
    if scheduler == "cosine":
        p = (step - warmup) / max(1, total - warmup)
        return base_lr * 0.5 * (1 + math.cos(math.pi * p))
    return base_lr

print("Modelos NOVEL + SE-Net + CBAM prontos.")

In [ ]:
# @title 4. Loop de treino + calibração por orçamento total
BATCH = 256
N_TRAIN = X_TRAIN.size(0)

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for i in range(0, X_VAL.size(0), 2048):
        x, y = X_VAL[i:i+2048], Y_VAL[i:i+2048]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
    return correct / X_VAL.size(0)

def train_one(mixing="none", attention="none", label_smoothing=0.0,
              warmup=0, scheduler="none", steps=2000, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    random.seed(seed)
    model = CIFARSmallCNN(mixing=mixing, attention=attention).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_acc = 0.0
    eval_every = max(1, steps // 4)
    t0 = time.perf_counter()
    model.train()
    for step in range(steps):
        idx = torch.randint(0, N_TRAIN, (BATCH,), device=DEVICE)
        x = X_TRAIN[idx].to(memory_format=torch.channels_last)
        y = Y_TRAIN[idx]
        for pg in opt.param_groups:
            pg["lr"] = lr_at_step(lr, step, steps, warmup, scheduler)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = F.cross_entropy(model(x), y, label_smoothing=label_smoothing)
        loss.backward()
        opt.step()
        if (step + 1) % eval_every == 0 or step == steps - 1:
            best_acc = max(best_acc, evaluate(model))
            model.train()
    return best_acc, time.perf_counter() - t0

# calibra no config mais pesado (tri_scale)
print("Calibrando GPU (200 steps, tri_scale)...")
_, t_cal = train_one(mixing="tri_scale_blend", steps=200, seed=0)
sps = 200 / t_cal
print(f"~{sps:.1f} steps/s")

per_run_sec = (TARGET_MINUTES * 60) / (N_CONFIGS * N_SEEDS)
STEPS = int(min(6000, max(400, sps * per_run_sec * 0.85)))  # 85% do budget (eval consome tempo)
print(f"Orçamento: {TARGET_MINUTES} min total | ~{per_run_sec:.0f}s/treino | STEPS={STEPS}")
print(f"≈ {STEPS * BATCH / 50000:.0f} épocas/treino | ETA total ~{TARGET_MINUTES} min")

In [ ]:
# @title 5. Head-to-head: NOVEL ATLAS vs SE-Net vs CBAM (3 seeds)
from statistics import mean, stdev

CONFIGS = {
    # régua
    "baseline":          dict(mixing="none", attention="none"),
    # NOVEL ATLAS
    "tri_scale_ls_wc":   dict(mixing="tri_scale_blend", label_smoothing=0.1, warmup=200, scheduler="cosine"),
    "multi_scale_ls_wc": dict(mixing="multi_scale_blend", label_smoothing=0.1, warmup=200, scheduler="cosine"),
    "blend_ls_wc":       dict(mixing="local_blend", label_smoothing=0.1, warmup=200, scheduler="cosine"),
    # LITERATURA
    "se_block":          dict(attention="se"),
    "se_ls_wc":          dict(attention="se", label_smoothing=0.1, warmup=200, scheduler="cosine"),
    "cbam":              dict(attention="cbam"),
    "cbam_ls_wc":        dict(attention="cbam", label_smoothing=0.1, warmup=200, scheduler="cosine"),
}

SEEDS = [1000, 1001, 1002]
results = {}
t0 = time.perf_counter()

for name, kw in CONFIGS.items():
    accs = []
    for seed in SEEDS:
        acc, _ = train_one(steps=STEPS, seed=seed, **kw)
        accs.append(acc)
    results[name] = (mean(accs), stdev(accs), accs)
    elapsed = (time.perf_counter() - t0) / 60
    tag = "NOVEL" if "scale" in name or name == "blend_ls_wc" else ("LIT" if "se" in name or "cbam" in name else "BASE")
    print(f"[{elapsed:5.1f}m] [{tag:4}] {name:20} {mean(accs)*100:.2f}% ± {stdev(accs)*100:.2f}%  seeds={[f'{a*100:.1f}' for a in accs]}")

print(f"\nTempo total: {(time.perf_counter()-t0)/60:.1f} min")

In [ ]:
# @title 6. Ranking + veredito vs baseline + head-to-head NOVEL vs literatura
base_m, base_s, _ = results["baseline"]
noise = 2 * base_s

# melhor NOVEL e melhor LITERATURA
novel_names = ["tri_scale_ls_wc", "multi_scale_ls_wc", "blend_ls_wc"]
lit_names = ["se_block", "se_ls_wc", "cbam", "cbam_ls_wc"]
best_novel = max(novel_names, key=lambda n: results[n][0])
best_lit = max(lit_names, key=lambda n: results[n][0])

print("=" * 82)
print(f"{'RANKING':22} {'ACC':>8} {'±STD':>7} {'Δ BASE':>9}  {'TIPO':6}  VEREDITO")
print("=" * 82)
for name, (m, s, _) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    delta = (m - base_m) * 100
    tipo = "NOVEL" if name in novel_names else ("LIT" if name in lit_names else "BASE")
    if name == "baseline":
        v = "(régua)"
    elif delta > noise * 100:
        v = "CONFIRMADA"
    elif delta > 0:
        v = "marginal"
    else:
        v = "REFUTADA"
    print(f"{name:22} {m*100:7.2f}% {s*100:6.2f}% {delta:+8.2f}pp  {tipo:6}  {v}")

print("=" * 82)
print(f"\nBaseline: {base_m*100:.2f}% ± {base_s*100:.2f}% | limiar: {noise*100:.2f}pp")
print(f"GPU: {gpu} | STEPS={STEPS} | batch={BATCH}")

# --- head-to-head principal ---
bn, (bm, bs, _) = best_novel, results[best_novel]
ln, (lm, ls, _) = best_lit, results[best_lit]
delta_lit = (bm - lm) * 100

print("\n" + "=" * 82)
print("HEAD-TO-HEAD PRINCIPAL")
print("=" * 82)
print(f"  Melhor NOVEL ATLAS:  {best_novel:20} {bm*100:.2f}% ± {bs*100:.2f}%")
print(f"  Melhor LITERATURA:   {best_lit:20} {lm*100:.2f}% ± {ls*100:.2f}%")
print(f"  Δ NOVEL vs LIT:      {delta_lit:+.2f}pp")

if delta_lit > 0.3:
    print("\n✓ NOVEL SUPEROU LITERATURA em CIFAR-10 — generaliza além do FashionMNIST")
elif delta_lit > -0.3:
    print("\n~ EMPATE — NOVEL ≈ literatura; inconclusivo neste budget")
else:
    print("\n✗ LITERATURA SUPEROU NOVEL — blend não generaliza para CIFAR-10 neste setup")

print("\nComo interpretar:")
print("  CONFIRMADA  = ganho > 2× ruído do baseline (significativo neste teste)")
print("  marginal    = ganho pequeno, pode ser ruído de seed")
print("  REFUTADA    = pior que baseline neste budget")
print("\nCole a saída desta célula de volta no ATLAS para registrar no experiments_log.")